# **Food Price Data Collection and Inspection**

This notebook explores Kenya food price data as a potential additional predictor for the drought and food security early-warning model.

The current model uses rainfall and NDVI vegetation indicators. However, food insecurity is also affected by market conditions, especially staple food prices.

The goal of this notebook is to inspect food price data, understand its structure, clean it, and prepare it for future merging with the master modeling dataset.

Possible food price indicators include:

- maize prices
- bean prices
- sorghum prices
- wheat prices
- rice prices
- price rolling averages
- price changes over time
- price anomalies

This notebook focuses on data collection and preparation. The modeling step will come later after the food price data is cleaned and aligned with the IPC analysis periods.

## **Notebook Goal**

The goal of this notebook is to answer these questions:

1. What food price data is available for Kenya?
2. Which commodities are included?
3. Which markets are included?
4. What date range does the dataset cover?
5. Are the prices monthly, weekly, or irregular?
6. Can the data be converted into county-level or national-level features?
7. Can the food price data be merged with the existing IPC + rainfall + NDVI master dataset?

This step is important because food price data may not align perfectly with county-level IPC records. We first need to inspect the dataset before deciding how to use it in the model.

In [14]:
# Download Kenya food price data from HDX / WFP

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Project paths
RAW_DIR = Path("../02_data/raw") 
FOOD_PRICE_RAW_DIR = RAW_DIR / "food_prices"
FOOD_PRICE_RAW_DIR.mkdir(parents=True, exist_ok=True)

# Direct HDX CSV download link for Kenya WFP food prices
food_price_url = "https://data.humdata.org/dataset/e0d3fba6-f9a2-45d7-b949-140c455197ff/resource/517ee1bf-2437-4f8c-aa1b-cb9925b9d437/download/wfp_food_prices_ken.csv"

food_price_file = FOOD_PRICE_RAW_DIR / "wfp_food_prices_ken.csv"

# Download and save locally
food_prices = pd.read_csv(food_price_url)
food_prices.to_csv(food_price_file, index=False)

print("Food price data downloaded successfully.")
print("Saved to:", food_price_file)
print("Shape:", food_prices.shape)

food_prices.head()

Food price data downloaded successfully.
Saved to: ..\02_data\raw\food_prices\wfp_food_prices_ken.csv
Shape: (26745, 16)


,date,admin1,admin2,market,market_id,latitude,longitude,category,commodity,commodity_id,unit,priceflag,pricetype,currency,price,usdprice
0,2006-01-15,Coast,Mombasa,Mombasa,191,-4.05,39.67,cereals and tubers,Maize,51,KG,actual,Wholesale,KES,16.13,0.22
1,2006-01-15,Coast,Mombasa,Mombasa,191,-4.05,39.67,cereals and tubers,Maize (white),67,90 KG,actual,Wholesale,KES,1480.00,20.58
2,2006-01-15,Coast,Mombasa,Mombasa,191,-4.05,39.67,pulses and nuts,Beans,50,KG,actual,Wholesale,KES,33.63,0.47
3,2006-01-15,Coast,Mombasa,Mombasa,191,-4.05,39.67,pulses and nuts,Beans (dry),262,90 KG,actual,Wholesale,KES,3246.00,45.15
4,2006-01-15,Eastern,Kitui,Kitui,187,-1.37,38.02,cereals and tubers,Maize (white),67,KG,actual,Retail,KES,17.00,0.24


In [15]:
# Target counties used in our project
target_counties = [
    "Baringo", "Embu", "Garissa", "Isiolo", "Kajiado", "Kilifi",
    "Kitui", "Kwale", "Laikipia", "Lamu", "Makueni", "Mandera",
    "Marsabit", "Meru", "Narok", "Nyeri", "Samburu", "Taita Taveta",
    "Tana River", "Tharaka Nithi", "Turkana", "Wajir", "West Pokot"
]

# Check available markets in the food price dataset
markets = sorted(food_prices["market"].dropna().unique())

print("Number of markets in dataset:", len(markets))
print("\nMarkets:")
for market in markets:
    print(market)

Number of markets in dataset: 226

Markets:
Adele Center
Alango Arba
Alemsekon
Alinjugur
Alungu
Amaya
Aposta
Archers Post
Ashabito
Bangale
Bangladesh (Mombasa)
Banissa
Baragoi
Baraki
Benane
Bilbil
Biliqo
Boji
Boka
Bulesa Bulesa
Buna
Bura
Bute
Charidende
Chemolingot
Dadaab town
Dagahaley (Daadab)
Damajale
Dambala Fachana
Dandora (Nairobi)
Dertu
Dirib
Dub Goba
Dzikunze
El Wak
Eldas
Eldoret town (Uasin Gishu)
Elelea
Eliye Centre
Emali
Ethiopia (Kakuma)
Garbatulla
Garissa
Garissa town (Garissa)
Garsen
Griftu
Habaswein
Hadado
Hagadera (Daadab)
Hara (Garissa)
Hola (Tana River)
Hola town
HongKong (Kakuma)
IFO (Daadab)
Illbissil Food Market (Kajiado)
Iresaboru Centre
Isiolo town
Jarajara
Junda (Mombasa)
Kaakelai
Kaanwa (Tharaka Nithi)
Kachororoni
Kaembeni
Kaeris
Kaikor
Kajiado
Kakuma 2
Kakuma 3
Kakuma 4
Kakuma town
Kalahari (Mombasa)
Kalemgorok
Kalemunyang
Kaleng
Kalobeyei (Village 1)
Kalobeyei (Village 2)
Kalobeyei (Village 3)
Kalobeyei town
Kalokol
Kang'akipur
Kangemi (Nairobi)
Karare
Karati

## **Market-to-County Mapping**

The WFP food price dataset is market-level, while the project master dataset is county-level.

To make the food price data usable for modeling, we need to map markets to the 23 target ASAL counties used in this project.

Some markets clearly include the county name, such as:

- `Garissa town (Garissa)`
- `Lodwar (Turkana)`
- `Marigat (Baringo)`
- `Wote town (Makueni)`
- `Vanga (Kwale)`

Other markets do not include the county name directly, so a manual mapping is needed.

This mapping will allow us to create county-level monthly food price features where market coverage exists. For counties without direct market coverage, national monthly food price averages can later be used as proxy indicators.

In [24]:
import re

# Manual market-to-county mapping for target ASAL counties
market_county_keywords = {
    "Baringo": [
        "Marigat", "Kimalel", "Chemolingot", "Amaya", "Loboi"
    ],
    "Garissa": [
        "Garissa", "Dadaab", "Daadab", "Hagadera", "IFO", "Liboi",
        "Hara", "Dertu", "Masalani"
    ],
    "Isiolo": [
        "Isiolo", "Garbatulla", "Kinna", "Merti", "Oldonyiro",
        "Sericho", "Ngaremara"
    ],
    "Kajiado": [
        "Kajiado", "Kitengela", "Illbissil"
    ],
    "Kilifi": [
        "Kilifi", "Dzikunze", "Kaembeni"
    ],
    "Kitui": [
        "Kitui"
    ],
    "Kwale": [
        "Kwale", "Vanga"
    ],
    "Makueni": [
        "Makueni", "Wote", "Kathonzweni", "Kibwezi", "Mtito Andei", "Emali"
    ],
    "Mandera": [
        "Mandera", "Takaba", "El Wak", "Rhamu", "Banissa", "Ashabito"
    ],
    "Marsabit": [
        "Marsabit", "Moyale", "North Horr", "Laisamis", "Sololo",
        "Loglogo", "Karare", "Korr", "Maikona", "Kargi",
        "Loyangalani", "Merille"
    ],
    "Nyeri": [
        "Nyeri", "Karatina"
    ],
    "Samburu": [
        "Samburu", "Maralal", "Baragoi", "Wamba", "Archers Post",
        "Tuum", "Porro", "Loosuk", "Ngurunit"
    ],
    "Tana River": [
        "Tana River", "Hola", "Garsen", "Kipini", "Bangale",
        "Bura", "Wayu", "Tarasaa"
    ],
    "Tharaka Nithi": [
        "Tharaka Nithi", "Tharaka", "Kaanwa", "Kaanwa (Tharaka Nithi)"
    ],
    "Turkana": [
        "Turkana", "Lodwar", "Kakuma", "Kalobeyei", "Lokichar",
        "Lokichoggio", "Lokitaung", "Kalokol", "Lokori", "Lokiriama",
        "Lokangae", "Lokapel", "Lorugum", "Eliye", "Turkwel",
        "Todonyang", "Lowarengak"
    ],
    "Wajir": [
        "Wajir", "Eldas", "Griftu", "Habaswein", "Tarbaj",
        "Bute", "Buna", "Hadado", "Maalamin"
    ],
    "West Pokot": [
        "West Pokot", "Lomut", "Makutano"
    ]
}


def assign_county_from_market(market_name):
    market_name = str(market_name).lower().strip()
    
    for county, keywords in market_county_keywords.items():
        for keyword in keywords:
            # Use word boundaries so "hara" does NOT match inside "tharaka"
            pattern = r'\b' + re.escape(keyword.lower().strip()) + r'\b'
            if re.search(pattern, market_name):
                return county
    
    return "Unmapped"


# Re-apply mapping
food_prices_mapped = food_prices.copy()
food_prices_mapped["mapped_county"] = food_prices_mapped["market"].apply(assign_county_from_market)

# Check Tharaka Nithi specifically
tharaka_check = food_prices_mapped[
    food_prices_mapped["market"].astype(str).str.contains("Kaanwa|Tharaka", case=False, na=False)
][["market", "mapped_county"]].drop_duplicates()

print("Tharaka / Kaanwa mapping check:")
print(tharaka_check)

# Re-check target county coverage
mapped_target_counties = sorted(
    food_prices_mapped.loc[
        food_prices_mapped["mapped_county"].isin(target_counties),
        "mapped_county"
    ].unique()
)

missing_target_counties = [
    county for county in target_counties
    if county not in mapped_target_counties
]

print("\nTarget counties covered:", len(mapped_target_counties))
print(mapped_target_counties)

print("\nTarget counties missing:", len(missing_target_counties))
print(missing_target_counties)

print("\nMapped rows:", (food_prices_mapped["mapped_county"] != "Unmapped").sum())
print("Unmapped rows:", (food_prices_mapped["mapped_county"] == "Unmapped").sum())

Tharaka / Kaanwa mapping check:
                      market  mapped_county
6853  Kaanwa (Tharaka Nithi)  Tharaka Nithi

Target counties covered: 17
['Baringo', 'Garissa', 'Isiolo', 'Kajiado', 'Kilifi', 'Kitui', 'Kwale', 'Makueni', 'Mandera', 'Marsabit', 'Nyeri', 'Samburu', 'Tana River', 'Tharaka Nithi', 'Turkana', 'Wajir', 'West Pokot']

Target counties missing: 6
['Embu', 'Laikipia', 'Lamu', 'Meru', 'Narok', 'Taita Taveta']

Mapped rows: 15366
Unmapped rows: 11379


## **Food Price County Coverage Result**

The WFP Kenya food price dataset contains market-level food price records.

After mapping markets to the project's target ASAL counties, the dataset provides direct market coverage for 17 out of 23 target counties.

The counties without direct mapped market coverage are:

- Embu
- Laikipia
- Lamu
- Meru
- Narok
- Taita Taveta

This means food price data can still be used, but with care. County-level food price features will be created for counties with mapped market coverage. For counties without direct market coverage, national monthly food price averages will later be used as proxy indicators.

This hybrid approach keeps all target counties in the analysis while still using local market information where available.

In [25]:
# Clean food price data for feature creation

food_prices_clean = food_prices_mapped.copy()

# Convert date column
food_prices_clean["date"] = pd.to_datetime(food_prices_clean["date"], errors="coerce")

# Create monthly date
food_prices_clean["month"] = food_prices_clean["date"].dt.to_period("M").dt.to_timestamp()

# Convert price to numeric
food_prices_clean["price"] = pd.to_numeric(food_prices_clean["price"], errors="coerce")

# Keep valid rows only
food_prices_clean = food_prices_clean.dropna(
    subset=["month", "commodity", "market", "price"]
)

print("Cleaned food price shape:", food_prices_clean.shape)

food_prices_clean.head()

Cleaned food price shape: (26745, 18)


,date,admin1,admin2,market,market_id,latitude,longitude,category,commodity,commodity_id,unit,priceflag,pricetype,currency,price,usdprice,mapped_county,month
0,2006-01-15,Coast,Mombasa,Mombasa,191,-4.05,39.67,cereals and tubers,Maize,51,KG,actual,Wholesale,KES,16.13,0.22,Unmapped,2006-01-01
1,2006-01-15,Coast,Mombasa,Mombasa,191,-4.05,39.67,cereals and tubers,Maize (white),67,90 KG,actual,Wholesale,KES,1480.00,20.58,Unmapped,2006-01-01
2,2006-01-15,Coast,Mombasa,Mombasa,191,-4.05,39.67,pulses and nuts,Beans,50,KG,actual,Wholesale,KES,33.63,0.47,Unmapped,2006-01-01
3,2006-01-15,Coast,Mombasa,Mombasa,191,-4.05,39.67,pulses and nuts,Beans (dry),262,90 KG,actual,Wholesale,KES,3246.00,45.15,Unmapped,2006-01-01
4,2006-01-15,Eastern,Kitui,Kitui,187,-1.37,38.02,cereals and tubers,Maize (white),67,KG,actual,Retail,KES,17.00,0.24,Kitui,2006-01-01


In [26]:
# Check available commodities

commodity_counts = food_prices_clean["commodity"].value_counts()

print("Number of unique commodities:", food_prices_clean["commodity"].nunique())

commodity_counts.head(60)

Number of unique commodities: 51


commodity
Beans (dry)                   1804
Maize (white)                 1758
Maize                         1435
Salt                          1333
Sugar                         1325
Potatoes (Irish)              1321
Wheat flour                   1291
Beans                         1248
Sorghum                       1063
Rice (aromatic)                960
Maize (white, dry)             916
Oil (vegetable)                844
Milk (cow, pasteurized)        821
Kale                           757
Oil (vegetable, fortified)     736
Maize flour                    694
Rice                           688
Maize flour (white)            685
Milk (UHT)                     571
Bananas                        534
Tomatoes                       501
Cabbage                        478
Onions (dry)                   463
Potatoes (Irish, white)        387
Meat (goat)                    386
Beans (yellow)                 372
Meat (beef)                    342
Spinach                        295
Meat (came

In [27]:
# Show all unique commodities alphabetically

sorted(food_prices_clean["commodity"].dropna().unique())

['Bananas',
 'Beans',
 'Beans (dolichos)',
 'Beans (dry)',
 'Beans (kidney)',
 'Beans (mung)',
 'Beans (rosecoco)',
 'Beans (yellow)',
 'Bread',
 'Cabbage',
 'Cooking fat',
 'Cowpea leaves',
 'Cowpeas',
 'Cowpeas (dry)',
 'Fish (omena, dry)',
 'Fuel (diesel)',
 'Fuel (kerosene)',
 'Fuel (petrol-gasoline)',
 'Kale',
 'Maize',
 'Maize (white)',
 'Maize (white, dry)',
 'Maize flour',
 'Maize flour (white)',
 'Meat (beef)',
 'Meat (camel)',
 'Meat (goat)',
 'Milk (UHT)',
 'Milk (camel, fresh)',
 'Milk (cow, fresh)',
 'Milk (cow, pasteurized)',
 'Millet (finger)',
 'Oil (vegetable)',
 'Oil (vegetable, fortified)',
 'Onions (dry)',
 'Onions (red)',
 'Pigeon peas (dry)',
 'Potatoes (Irish)',
 'Potatoes (Irish, red)',
 'Potatoes (Irish, white)',
 'Rice',
 'Rice (aromatic)',
 'Rice (imported, Pakistan)',
 'Salt',
 'Sorghum',
 'Sorghum (red)',
 'Sorghum (white)',
 'Spinach',
 'Sugar',
 'Tomatoes',
 'Wheat flour']

## **Staple Food Commodity Selection**

The WFP Kenya food price dataset contains many commodities, including cereals, pulses, vegetables, meat, milk, fuel, salt, and sugar.

For this project, the first food price feature version will focus on staple foods that are more directly connected to household food security and drought response.

Selected staple commodity groups:

- Maize
- Maize flour
- Beans
- Cowpeas
- Sorghum
- Rice
- Wheat flour
- Millet

These commodities are useful because they represent common staple foods or substitute foods used by households during food stress periods.

Other commodities such as vegetables, meat, milk, fuel, salt, and sugar are not used in the first version to keep the feature engineering simple and focused.

In [28]:
# Create simplified staple commodity groups

def classify_staple_commodity(commodity):
    commodity = str(commodity).lower().strip()
    
    if "maize flour" in commodity:
        return "maize_flour"
    elif "maize" in commodity:
        return "maize"
    elif "beans" in commodity:
        return "beans"
    elif "cowpeas" in commodity or "cowpea" in commodity:
        return "cowpeas"
    elif "sorghum" in commodity:
        return "sorghum"
    elif "rice" in commodity:
        return "rice"
    elif "wheat flour" in commodity:
        return "wheat_flour"
    elif "millet" in commodity:
        return "millet"
    else:
        return "other"


food_prices_clean["staple_group"] = food_prices_clean["commodity"].apply(classify_staple_commodity)

# Check staple grouping result
food_prices_clean["staple_group"].value_counts()

staple_group
other          12342
maize           4109
beans           4104
rice            1668
maize_flour     1379
wheat_flour     1291
sorghum         1233
cowpeas          542
millet            77
Name: count, dtype: int64

## **Staple Food Coverage Check**

The selected staple foods do not have equal record counts.

Before creating modeling features, we need to check whether each staple food has enough coverage across:

1. Time
2. Markets
3. Mapped target counties

This is important because sparse commodities can create many missing values after merging with the IPC master dataset.

For the first modeling version, we will prioritize staple foods with stronger coverage, especially maize, beans, rice, wheat flour, maize flour, and sorghum.

In [29]:
# Check coverage for each staple food group

staple_coverage_summary = (
    food_prices_clean
    .groupby("staple_group")
    .agg(
        records=("price", "count"),
        first_month=("month", "min"),
        last_month=("month", "max"),
        markets=("market", "nunique"),
        mapped_counties=("mapped_county", lambda x: x[x != "Unmapped"].nunique()),
        unmapped_rows=("mapped_county", lambda x: (x == "Unmapped").sum())
    )
    .reset_index()
    .sort_values("records", ascending=False)
)

staple_coverage_summary

,staple_group,records,first_month,last_month,markets,mapped_counties,unmapped_rows
5,other,12342,2006-01-01,2026-04-01,221,17,4696
2,maize,4109,2006-01-01,2026-04-01,201,16,2018
0,beans,4104,2006-01-01,2026-04-01,210,16,2297
6,rice,1668,2018-12-01,2026-04-01,212,16,577
3,maize_flour,1379,2018-12-01,2026-04-01,199,11,417
8,wheat_flour,1291,2018-12-01,2026-04-01,199,11,412
7,sorghum,1233,2006-01-01,2026-04-01,46,12,721
1,cowpeas,542,2021-01-01,2026-04-01,133,16,214
4,millet,77,2021-01-01,2021-11-01,13,7,27


## **Staple Food Coverage Summary**

The commodity coverage check shows that the WFP Kenya food price dataset contains several staple food groups, but they do not all have equal coverage.

For the first food price feature version, the strongest candidates are:

- `maize`
- `beans`
- `rice`

These three commodities have strong record counts, broad market coverage, and useful time coverage for the project period.

From the coverage table:

- `maize` has 4,109 records and covers 16 mapped target counties.
- `beans` has 4,104 records and covers 16 mapped target counties.
- `rice` has 1,668 records and covers 16 mapped target counties.

Other commodities such as `maize_flour`, `wheat_flour`, `sorghum`, and `cowpeas` are useful, but they have weaker county coverage or fewer records. They can be tested later as optional features.

`millet` has very limited coverage, with only 77 records, 7 mapped counties, and data mainly within 2021. Because of this, millet will not be included in the first modeling feature version.

For this notebook, the first feature engineering version will focus on maize, beans, and rice. These will be used to create both county-level and national-level monthly food price indicators.

## **Main Staple Food Selection**

Based on the commodity coverage check, the first food price feature version will use only three main staple groups:

- `maize`
- `beans`
- `rice`

These commodities were selected because they have strong record counts, good mapped-county coverage, and meaningful food security relevance.

The goal is to keep the first food price feature version simple, clean, and stable before testing additional commodities later.

The selected staple foods will be used to create:

1. County-level monthly food price features for counties with mapped market coverage.
2. National monthly food price features that can be used as proxy indicators for counties without direct market coverage.

In [30]:
# Select main staple foods for first food price feature version

main_staple_groups = ["maize", "beans", "rice"]

food_prices_main_staples = food_prices_clean[
    food_prices_clean["staple_group"].isin(main_staple_groups)
].copy()

print("Main staple food price shape:", food_prices_main_staples.shape)

food_prices_main_staples[
    ["month", "market", "mapped_county", "commodity", "staple_group", "unit", "pricetype", "price"]
].head(20)

Main staple food price shape: (9881, 19)


,month,market,mapped_county,commodity,staple_group,unit,pricetype,price
0,2006-01-01,Mombasa,Unmapped,Maize,maize,KG,Wholesale,16.13
1,2006-01-01,Mombasa,Unmapped,Maize (white),maize,90 KG,Wholesale,1480.00
2,2006-01-01,Mombasa,Unmapped,Beans,beans,KG,Wholesale,33.63
3,2006-01-01,Mombasa,Unmapped,Beans (dry),beans,90 KG,Wholesale,3246.00
4,2006-01-01,Kitui,Kitui,Maize (white),maize,KG,Retail,17.00
7,2006-01-01,Kitui,Kitui,Beans (dry),beans,KG,Retail,39.00
8,2006-01-01,Marsabit,Marsabit,Maize (white),maize,KG,Retail,21.00
10,2006-01-01,Nairobi,Unmapped,Maize,maize,KG,Wholesale,15.48
11,2006-01-01,Nairobi,Unmapped,Maize (white),maize,90 KG,Wholesale,1399.00
16,2006-01-01,Nairobi,Unmapped,Beans,beans,KG,Wholesale,42.31


In [33]:
# Check which original commodities are included inside each selected staple group

food_prices_clean[
    food_prices_clean["staple_group"].isin(["maize", "beans", "rice"])
].groupby("staple_group")["commodity"].unique()

staple_group
beans    [Beans, Beans (dry), Beans (kidney), Beans (ro...
maize           [Maize, Maize (white), Maize (white, dry)]
rice     [Rice, Rice (aromatic), Rice (imported, Pakist...
Name: commodity, dtype: object

## **Price Unit Standardization**

The selected food price records use different units, such as `KG` and `90 KG`.

Before creating monthly average prices, all prices must be converted into a common unit.

For this notebook, prices will be standardized to estimated price per kilogram:

- If unit is `KG`, use the price as-is.
- If unit is `50KG`, divide the price by 50.
- If unit is `90 KG`, divide the price by 90.

This prevents large bag prices from being incorrectly averaged with single-kilogram prices.

In [31]:
# Check units used by the selected staple foods

food_prices_main_staples["unit"].value_counts()

unit
KG       6682
90 KG    2912
50 KG     287
Name: count, dtype: int64

In [32]:
# Create standardized price per KG

food_prices_main_staples = food_prices_main_staples.copy()

def convert_price_to_kg(row):
    unit = str(row["unit"]).upper().strip()
    price = row["price"]
    
    if unit == "KG":
        return price
    elif unit == "90 KG":
        return price / 90
    elif unit == "50 KG":
        return price / 50
    else:
        return np.nan

food_prices_main_staples["price_per_kg"] = food_prices_main_staples.apply(
    convert_price_to_kg,
    axis=1
)

# Drop rows where price per KG could not be calculated
food_prices_main_staples = food_prices_main_staples.dropna(subset=["price_per_kg"])

print("Shape after price-per-kg conversion:", food_prices_main_staples.shape)

food_prices_main_staples[
    ["month", "market", "mapped_county", "commodity", "staple_group", "unit", "price", "price_per_kg"]
].head(20)

Shape after price-per-kg conversion: (9881, 20)


,month,market,mapped_county,commodity,staple_group,unit,price,price_per_kg
0,2006-01-01,Mombasa,Unmapped,Maize,maize,KG,16.13,16.130000
1,2006-01-01,Mombasa,Unmapped,Maize (white),maize,90 KG,1480.00,16.444444
2,2006-01-01,Mombasa,Unmapped,Beans,beans,KG,33.63,33.630000
3,2006-01-01,Mombasa,Unmapped,Beans (dry),beans,90 KG,3246.00,36.066667
4,2006-01-01,Kitui,Kitui,Maize (white),maize,KG,17.00,17.000000
7,2006-01-01,Kitui,Kitui,Beans (dry),beans,KG,39.00,39.000000
8,2006-01-01,Marsabit,Marsabit,Maize (white),maize,KG,21.00,21.000000
10,2006-01-01,Nairobi,Unmapped,Maize,maize,KG,15.48,15.480000
11,2006-01-01,Nairobi,Unmapped,Maize (white),maize,90 KG,1399.00,15.544444
16,2006-01-01,Nairobi,Unmapped,Beans,beans,KG,42.31,42.310000


In [34]:
# Check standardized prices by staple group and unit

price_unit_check = (
    food_prices_main_staples
    .groupby(["staple_group", "unit"])
    .agg(
        records=("price_per_kg", "count"),
        avg_original_price=("price", "mean"),
        avg_price_per_kg=("price_per_kg", "mean"),
        min_price_per_kg=("price_per_kg", "min"),
        max_price_per_kg=("price_per_kg", "max")
    )
    .reset_index()
    .sort_values(["staple_group", "unit"])
)

price_unit_check

,staple_group,unit,records,avg_original_price,avg_price_per_kg,min_price_per_kg,max_price_per_kg
0,beans,90 KG,1785,8228.953720,91.432819,12.600000,220.000000
1,beans,KG,2319,107.992147,107.992147,23.000000,270.000000
2,maize,90 KG,1127,3233.466043,35.927400,8.911111,90.422222
3,maize,KG,2982,51.014614,51.014614,6.000000,152.000000
4,rice,50 KG,287,6842.771185,136.855424,75.000000,212.500000
5,rice,KG,1381,129.823338,129.823338,73.330000,245.000000


## **Price Per Kilogram Conversion Result**

The selected staple food records were successfully converted to estimated price per kilogram.

This step is important because the raw dataset contains different units such as `KG`, `50 KG`, and `90 KG`.

After conversion, the notebook will use the standardized `price_per_kg` column for all monthly price feature calculations.

The original `price` column will not be used for averaging because it mixes single-kilogram prices with large bag prices.

In [35]:
# Create county-level monthly average prices for main staple foods

county_monthly_prices_long = (
    food_prices_main_staples[
        food_prices_main_staples["mapped_county"].isin(target_counties)
    ]
    .groupby(["mapped_county", "month", "staple_group"], as_index=False)
    .agg(
        avg_price_per_kg=("price_per_kg", "mean"),
        min_price_per_kg=("price_per_kg", "min"),
        max_price_per_kg=("price_per_kg", "max"),
        market_count=("market", "nunique"),
        record_count=("price_per_kg", "count")
    )
)

print("County monthly prices long shape:", county_monthly_prices_long.shape)

county_monthly_prices_long.head()

County monthly prices long shape: (2423, 8)


,mapped_county,month,staple_group,avg_price_per_kg,min_price_per_kg,max_price_per_kg,market_count,record_count
0,Baringo,2015-01-01,beans,104.0,104.0,104.0,1,1
1,Baringo,2015-01-01,maize,42.0,42.0,42.0,1,1
2,Baringo,2015-02-01,beans,107.0,107.0,107.0,1,1
3,Baringo,2015-02-01,maize,47.0,47.0,47.0,1,1
4,Baringo,2015-03-01,beans,112.0,112.0,112.0,1,1


In [36]:
# Pivot county-level staple prices into wide modeling format

county_price_features = county_monthly_prices_long.pivot_table(
    index=["mapped_county", "month"],
    columns="staple_group",
    values="avg_price_per_kg",
    aggfunc="mean"
).reset_index()

# Rename columns
county_price_features = county_price_features.rename(columns={
    "mapped_county": "county",
    "maize": "county_maize_price_per_kg",
    "beans": "county_beans_price_per_kg",
    "rice": "county_rice_price_per_kg"
})

# Create overall county staple average price
county_price_cols = [
    "county_maize_price_per_kg",
    "county_beans_price_per_kg",
    "county_rice_price_per_kg"
]

county_price_features["county_staple_price_per_kg"] = county_price_features[county_price_cols].mean(axis=1)

# Add flag showing county-level price is available
county_price_features["county_price_available"] = 1

print("County price features shape:", county_price_features.shape)

county_price_features.head()

County price features shape: (1448, 7)


staple_group,county,month,county_beans_price_per_kg,county_maize_price_per_kg,county_rice_price_per_kg,county_staple_price_per_kg,county_price_available
0,Baringo,2015-01-01,104.0,42.0,NaN,73.0,1
1,Baringo,2015-02-01,107.0,47.0,NaN,77.0,1
2,Baringo,2015-03-01,112.0,38.0,NaN,75.0,1
3,Baringo,2015-04-01,105.0,42.0,NaN,73.5,1
4,Baringo,2015-05-01,106.0,45.0,NaN,75.5,1


In [37]:
# Check missing values in county price features

county_price_features.isna().sum()

staple_group
county                           0
month                            0
county_beans_price_per_kg      711
county_maize_price_per_kg       38
county_rice_price_per_kg      1172
county_staple_price_per_kg       0
county_price_available           0
dtype: int64

In [38]:
# Check percentage of missing values in county price features

missing_summary = (
    county_price_features
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .reset_index()
)

missing_summary.columns = ["column", "missing_percentage"]

missing_summary

,column,missing_percentage
0,county,0.00
1,month,0.00
2,county_beans_price_per_kg,49.10
3,county_maize_price_per_kg,2.62
4,county_rice_price_per_kg,80.94
5,county_staple_price_per_kg,0.00
6,county_price_available,0.00


## **County Price Feature Missingness Insight**

The county-level price feature table shows that individual commodity coverage is uneven.

`county_maize_price_per_kg` has the strongest coverage, with only 2.62% of missing values.

`county_beans_price_per_kg` has moderate missingness, while `county_rice_price_per_kg` has high missingness. This means rice is useful for the overall staple price signal, but it may be too sparse to use as a standalone county-level feature in the first model.

The combined feature `county_staple_price_per_kg` has no missing values because it averages the available staple food prices for each county-month.

For the first modeling version, `county_staple_price_per_kg` will be treated as the main county-level food price feature. `county_maize_price_per_kg` may also be useful because it has strong coverage.

In [39]:
# Count how many staple food prices are available per county-month

county_price_cols = [
    "county_maize_price_per_kg",
    "county_beans_price_per_kg",
    "county_rice_price_per_kg"
]

county_price_features["county_staple_items_available"] = (
    county_price_features[county_price_cols]
    .notna()
    .sum(axis=1)
)

county_price_features[
    [
        "county",
        "month",
        "county_maize_price_per_kg",
        "county_beans_price_per_kg",
        "county_rice_price_per_kg",
        "county_staple_price_per_kg",
        "county_staple_items_available",
        "county_price_available"
    ]
].head(20)

staple_group,county,month,county_maize_price_per_kg,county_beans_price_per_kg,county_rice_price_per_kg,county_staple_price_per_kg,county_staple_items_available,county_price_available
0,Baringo,2015-01-01,42.0,104.0,NaN,73.0,2,1
1,Baringo,2015-02-01,47.0,107.0,NaN,77.0,2,1
2,Baringo,2015-03-01,38.0,112.0,NaN,75.0,2,1
3,Baringo,2015-04-01,42.0,105.0,NaN,73.5,2,1
4,Baringo,2015-05-01,45.0,106.0,NaN,75.5,2,1
5,Baringo,2015-06-01,45.0,113.0,NaN,79.0,2,1
6,Baringo,2015-07-01,44.0,105.0,NaN,74.5,2,1
7,Baringo,2015-08-01,46.0,102.0,NaN,74.0,2,1
8,Baringo,2015-09-01,44.0,102.0,NaN,73.0,2,1
9,Baringo,2015-10-01,37.0,103.0,NaN,70.0,2,1


In [40]:
# Check how many county-month rows have 1, 2, or 3 staple foods available

county_price_features["county_staple_items_available"].value_counts().sort_index()

county_staple_items_available
1    730
2    461
3    257
Name: count, dtype: int64

## **County Staple Price Quality Check**

The county-level staple price feature does not always use all three selected commodities.

The output shows that:

- 730 county-month rows have 1 staple food price available.
- 461 county-month rows have 2 staple food prices available.
- 257 county-month rows have all 3 staple food prices available.

This means `county_staple_price_per_kg` is useful, but its quality differs by county and month.

To make this transparent, the notebook keeps the `county_staple_items_available` column as a quality flag. This column shows whether the county staple price average was calculated from 1, 2, or 3 available staple foods.

For the first modeling version, `county_staple_price_per_kg` will be the main county-level food price feature, while `county_staple_items_available` will help document data coverage quality.

## **County Food Price Rolling Average Features**

Food price changes may affect food insecurity over time, not only in the current month.

To capture short-term and medium-term market pressure, rolling average features are created for county-level prices.

This notebook creates:

- 3-month rolling averages
- 6-month rolling averages

These features follow the same idea used earlier for rainfall and NDVI features.

In [41]:
# Add rolling average features for county-level food prices

county_price_features = county_price_features.sort_values(["county", "month"]).copy()

rolling_cols = [
    "county_maize_price_per_kg",
    "county_beans_price_per_kg",
    "county_rice_price_per_kg",
    "county_staple_price_per_kg"
]

for col in rolling_cols:
    county_price_features[f"{col}_3_month_avg"] = (
        county_price_features
        .groupby("county")[col]
        .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    )
    
    county_price_features[f"{col}_6_month_avg"] = (
        county_price_features
        .groupby("county")[col]
        .transform(lambda x: x.rolling(window=6, min_periods=1).mean())
    )

print("County price features with rolling averages shape:", county_price_features.shape)

county_price_features.head()

County price features with rolling averages shape: (1448, 16)


staple_group,county,month,county_beans_price_per_kg,county_maize_price_per_kg,county_rice_price_per_kg,county_staple_price_per_kg,county_price_available,county_staple_items_available,county_maize_price_per_kg_3_month_avg,county_maize_price_per_kg_6_month_avg,county_beans_price_per_kg_3_month_avg,county_beans_price_per_kg_6_month_avg,county_rice_price_per_kg_3_month_avg,county_rice_price_per_kg_6_month_avg,county_staple_price_per_kg_3_month_avg,county_staple_price_per_kg_6_month_avg
0,Baringo,2015-01-01,104.0,42.0,NaN,73.0,1,2,42.000000,42.000000,104.000000,104.000000,NaN,NaN,73.000000,73.000
1,Baringo,2015-02-01,107.0,47.0,NaN,77.0,1,2,44.500000,44.500000,105.500000,105.500000,NaN,NaN,75.000000,75.000
2,Baringo,2015-03-01,112.0,38.0,NaN,75.0,1,2,42.333333,42.333333,107.666667,107.666667,NaN,NaN,75.000000,75.000
3,Baringo,2015-04-01,105.0,42.0,NaN,73.5,1,2,42.333333,42.250000,108.000000,107.000000,NaN,NaN,75.166667,74.625
4,Baringo,2015-05-01,106.0,45.0,NaN,75.5,1,2,41.666667,42.800000,107.666667,106.800000,NaN,NaN,74.666667,74.800


In [44]:
# Check missing values after rolling feature creation

county_price_features.isna().sum()

# Check missing values only during the IPC/master dataset analysis period

analysis_start = pd.to_datetime("2019-07-01")
analysis_end = pd.to_datetime("2026-02-01")

county_price_features_analysis_period = county_price_features[
    (county_price_features["month"] >= analysis_start) &
    (county_price_features["month"] <= analysis_end)
].copy()

print("County price features full shape:", county_price_features.shape)
print("County price features analysis-period shape:", county_price_features_analysis_period.shape)

county_price_features_analysis_period.isna().sum()

County price features full shape: (1448, 16)
County price features analysis-period shape: (540, 16)


staple_group
county                                      0
month                                       0
county_beans_price_per_kg                 125
county_maize_price_per_kg                  28
county_rice_price_per_kg                  269
county_staple_price_per_kg                  0
county_price_available                      0
county_staple_items_available               0
county_maize_price_per_kg_3_month_avg       5
county_maize_price_per_kg_6_month_avg       3
county_beans_price_per_kg_3_month_avg      98
county_beans_price_per_kg_6_month_avg      84
county_rice_price_per_kg_3_month_avg      244
county_rice_price_per_kg_6_month_avg      226
county_staple_price_per_kg_3_month_avg      0
county_staple_price_per_kg_6_month_avg      0
dtype: int64

In [45]:
# Missing value percentage during the analysis period

missing_summary_analysis_period = (
    county_price_features_analysis_period
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .reset_index()
)

missing_summary_analysis_period.columns = ["column", "missing_percentage"]

missing_summary_analysis_period

,column,missing_percentage
0,county,0.00
1,month,0.00
2,county_beans_price_per_kg,23.15
3,county_maize_price_per_kg,5.19
4,county_rice_price_per_kg,49.81
5,county_staple_price_per_kg,0.00
6,county_price_available,0.00
7,county_staple_items_available,0.00
8,county_maize_price_per_kg_3_month_avg,0.93
9,county_maize_price_per_kg_6_month_avg,0.56


## **Analysis-Period Missing Value Check**

The food price feature table was filtered to match the project analysis period from July 2019 to February 2026.

This is important because the raw food price dataset starts much earlier than the IPC master dataset. Checking missing values across the full food price history can give a misleading picture.

During the analysis period, the combined county staple price features have complete coverage:

- `county_staple_price_per_kg`
- `county_staple_price_per_kg_3_month_avg`
- `county_staple_price_per_kg_6_month_avg`

Individual commodity features have uneven coverage. Maize has strong coverage, beans has moderate coverage, and rice has weaker county-level coverage.

For the first modeling version, the combined staple price features will be treated as the main county-level food price indicators.

## **National Monthly Food Price Features**

County-level food price data is not available for all 23 target counties.

To keep full project coverage, national monthly food price features are created as backup/proxy indicators.

These national features use Kenya-wide monthly average prices for the selected staple groups:

- maize
- beans
- rice

The national features will later help fill the gap for counties without direct market coverage.

In [46]:
# Create national monthly average prices for main staple foods

national_monthly_prices_long = (
    food_prices_main_staples
    .groupby(["month", "staple_group"], as_index=False)
    .agg(
        avg_price_per_kg=("price_per_kg", "mean"),
        min_price_per_kg=("price_per_kg", "min"),
        max_price_per_kg=("price_per_kg", "max"),
        market_count=("market", "nunique"),
        record_count=("price_per_kg", "count")
    )
)

print("National monthly prices long shape:", national_monthly_prices_long.shape)

national_monthly_prices_long.head()

National monthly prices long shape: (555, 7)


,month,staple_group,avg_price_per_kg,min_price_per_kg,max_price_per_kg,market_count,record_count
0,2006-01-01,beans,38.587654,31.100000,45.85,5,9
1,2006-01-01,maize,17.775833,12.960000,30.00,8,12
2,2006-02-01,beans,41.259383,35.000000,47.10,5,9
3,2006-02-01,maize,18.741019,13.120000,29.00,8,12
4,2006-03-01,beans,45.745864,36.722222,54.52,5,9


In [47]:
# Pivot national staple prices into wide modeling format

national_price_features = national_monthly_prices_long.pivot_table(
    index="month",
    columns="staple_group",
    values="avg_price_per_kg",
    aggfunc="mean"
).reset_index()

# Rename columns
national_price_features = national_price_features.rename(columns={
    "maize": "national_maize_price_per_kg",
    "beans": "national_beans_price_per_kg",
    "rice": "national_rice_price_per_kg"
})

# Create overall national staple average price
national_price_cols = [
    "national_maize_price_per_kg",
    "national_beans_price_per_kg",
    "national_rice_price_per_kg"
]

national_price_features["national_staple_price_per_kg"] = (
    national_price_features[national_price_cols].mean(axis=1)
)

print("National price features shape:", national_price_features.shape)

national_price_features.head()

National price features shape: (244, 5)


staple_group,month,national_beans_price_per_kg,national_maize_price_per_kg,national_rice_price_per_kg,national_staple_price_per_kg
0,2006-01-01,38.587654,17.775833,NaN,28.181744
1,2006-02-01,41.259383,18.741019,NaN,30.000201
2,2006-03-01,45.745864,18.266917,NaN,32.006390
3,2006-04-01,46.724815,19.791111,NaN,33.257963
4,2006-05-01,47.074753,20.242859,NaN,33.658806


In [48]:
# Add rolling average features for national food prices

national_price_features = national_price_features.sort_values("month").copy()

national_rolling_cols = [
    "national_maize_price_per_kg",
    "national_beans_price_per_kg",
    "national_rice_price_per_kg",
    "national_staple_price_per_kg"
]

for col in national_rolling_cols:
    national_price_features[f"{col}_3_month_avg"] = (
        national_price_features[col]
        .rolling(window=3, min_periods=1)
        .mean()
    )
    
    national_price_features[f"{col}_6_month_avg"] = (
        national_price_features[col]
        .rolling(window=6, min_periods=1)
        .mean()
    )

print("National price features with rolling averages shape:", national_price_features.shape)

national_price_features.head()

National price features with rolling averages shape: (244, 13)


staple_group,month,national_beans_price_per_kg,national_maize_price_per_kg,national_rice_price_per_kg,national_staple_price_per_kg,national_maize_price_per_kg_3_month_avg,national_maize_price_per_kg_6_month_avg,national_beans_price_per_kg_3_month_avg,national_beans_price_per_kg_6_month_avg,national_rice_price_per_kg_3_month_avg,national_rice_price_per_kg_6_month_avg,national_staple_price_per_kg_3_month_avg,national_staple_price_per_kg_6_month_avg
0,2006-01-01,38.587654,17.775833,NaN,28.181744,17.775833,17.775833,38.587654,38.587654,NaN,NaN,28.181744,28.181744
1,2006-02-01,41.259383,18.741019,NaN,30.000201,18.258426,18.258426,39.923519,39.923519,NaN,NaN,29.090972,29.090972
2,2006-03-01,45.745864,18.266917,NaN,32.006390,18.261256,18.261256,41.864300,41.864300,NaN,NaN,30.062778,30.062778
3,2006-04-01,46.724815,19.791111,NaN,33.257963,18.933015,18.643720,44.576687,43.079429,NaN,NaN,31.754851,30.861574
4,2006-05-01,47.074753,20.242859,NaN,33.658806,19.433629,18.963548,46.515144,43.878494,NaN,NaN,32.974386,31.421021


In [49]:
# Check national price missingness during the analysis period

national_price_features_analysis_period = national_price_features[
    (national_price_features["month"] >= analysis_start) &
    (national_price_features["month"] <= analysis_end)
].copy()

print("National price features full shape:", national_price_features.shape)
print("National price features analysis-period shape:", national_price_features_analysis_period.shape)

national_price_features_analysis_period.isna().sum()

National price features full shape: (244, 13)
National price features analysis-period shape: (80, 13)


staple_group
month                                        0
national_beans_price_per_kg                  0
national_maize_price_per_kg                  0
national_rice_price_per_kg                  17
national_staple_price_per_kg                 0
national_maize_price_per_kg_3_month_avg      0
national_maize_price_per_kg_6_month_avg      0
national_beans_price_per_kg_3_month_avg      0
national_beans_price_per_kg_6_month_avg      0
national_rice_price_per_kg_3_month_avg      15
national_rice_price_per_kg_6_month_avg      13
national_staple_price_per_kg_3_month_avg     0
national_staple_price_per_kg_6_month_avg     0
dtype: int64

In [50]:
# Missing percentage for national price features during analysis period

national_missing_summary_analysis_period = (
    national_price_features_analysis_period
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .reset_index()
)

national_missing_summary_analysis_period.columns = ["column", "missing_percentage"]

national_missing_summary_analysis_period

,column,missing_percentage
0,month,0.00
1,national_beans_price_per_kg,0.00
2,national_maize_price_per_kg,0.00
3,national_rice_price_per_kg,21.25
4,national_staple_price_per_kg,0.00
5,national_maize_price_per_kg_3_month_avg,0.00
6,national_maize_price_per_kg_6_month_avg,0.00
7,national_beans_price_per_kg_3_month_avg,0.00
8,national_beans_price_per_kg_6_month_avg,0.00
9,national_rice_price_per_kg_3_month_avg,18.75


## **National Food Price Feature Results**

National monthly food price features were created successfully.

The national feature table contains one row per month and includes maize, beans, rice, and combined staple price indicators.

During the project analysis period from July 2019 to February 2026, the main national food price features have complete coverage:

- `national_maize_price_per_kg`
- `national_beans_price_per_kg`
- `national_staple_price_per_kg`
- `national_staple_price_per_kg_3_month_avg`
- `national_staple_price_per_kg_6_month_avg`

Rice still has some missing values, so it should be used carefully as an individual feature.

The combined national staple price features are the safest national market indicators for the first modeling version.

## **Save Processed Food Price Feature Files**

The cleaned and engineered food price features will now be saved as processed datasets.

Two output files are created:

1. `county_food_price_monthly_features.csv`  
   Contains county-level monthly food price features for counties with mapped market coverage.

2. `national_food_price_monthly_features.csv`  
   Contains Kenya-wide monthly food price features that can be used as proxy indicators for all counties.

These files will be used later when merging food price indicators with the IPC + rainfall + NDVI master modeling dataset.

In [51]:
# Save processed food price feature files

county_price_output = PROCESSED_DIR / "county_food_price_monthly_features.csv"
national_price_output = PROCESSED_DIR / "national_food_price_monthly_features.csv"

county_price_features.to_csv(county_price_output, index=False)
national_price_features.to_csv(national_price_output, index=False)

print("County food price features saved to:")
print(county_price_output)
print("Shape:", county_price_features.shape)

print("\nNational food price features saved to:")
print(national_price_output)
print("Shape:", national_price_features.shape)

County food price features saved to:
..\02_data\processed\county_food_price_monthly_features.csv
Shape: (1448, 16)

National food price features saved to:
..\02_data\processed\national_food_price_monthly_features.csv
Shape: (244, 13)


In [52]:
# Verify saved processed files

saved_county_prices = pd.read_csv(county_price_output)
saved_national_prices = pd.read_csv(national_price_output)

print("Saved county price file shape:", saved_county_prices.shape)
print("Saved national price file shape:", saved_national_prices.shape)

display(saved_county_prices.head())
display(saved_national_prices.head())

Saved county price file shape: (1448, 16)
Saved national price file shape: (244, 13)


,county,month,county_beans_price_per_kg,county_maize_price_per_kg,county_rice_price_per_kg,county_staple_price_per_kg,county_price_available,county_staple_items_available,county_maize_price_per_kg_3_month_avg,county_maize_price_per_kg_6_month_avg,county_beans_price_per_kg_3_month_avg,county_beans_price_per_kg_6_month_avg,county_rice_price_per_kg_3_month_avg,county_rice_price_per_kg_6_month_avg,county_staple_price_per_kg_3_month_avg,county_staple_price_per_kg_6_month_avg
0,Baringo,2015-01-01,104.0,42.0,NaN,73.0,1,2,42.000000,42.000000,104.000000,104.000000,NaN,NaN,73.000000,73.000
1,Baringo,2015-02-01,107.0,47.0,NaN,77.0,1,2,44.500000,44.500000,105.500000,105.500000,NaN,NaN,75.000000,75.000
2,Baringo,2015-03-01,112.0,38.0,NaN,75.0,1,2,42.333333,42.333333,107.666667,107.666667,NaN,NaN,75.000000,75.000
3,Baringo,2015-04-01,105.0,42.0,NaN,73.5,1,2,42.333333,42.250000,108.000000,107.000000,NaN,NaN,75.166667,74.625
4,Baringo,2015-05-01,106.0,45.0,NaN,75.5,1,2,41.666667,42.800000,107.666667,106.800000,NaN,NaN,74.666667,74.800


,month,national_beans_price_per_kg,national_maize_price_per_kg,national_rice_price_per_kg,national_staple_price_per_kg,national_maize_price_per_kg_3_month_avg,national_maize_price_per_kg_6_month_avg,national_beans_price_per_kg_3_month_avg,national_beans_price_per_kg_6_month_avg,national_rice_price_per_kg_3_month_avg,national_rice_price_per_kg_6_month_avg,national_staple_price_per_kg_3_month_avg,national_staple_price_per_kg_6_month_avg
0,2006-01-01,38.587654,17.775833,NaN,28.181744,17.775833,17.775833,38.587654,38.587654,NaN,NaN,28.181744,28.181744
1,2006-02-01,41.259383,18.741019,NaN,30.000201,18.258426,18.258426,39.923519,39.923519,NaN,NaN,29.090972,29.090972
2,2006-03-01,45.745864,18.266917,NaN,32.006390,18.261256,18.261256,41.864300,41.864300,NaN,NaN,30.062778,30.062778
3,2006-04-01,46.724815,19.791111,NaN,33.257963,18.933015,18.643720,44.576687,43.079429,NaN,NaN,31.754851,30.861574
4,2006-05-01,47.074753,20.242859,NaN,33.658806,19.433629,18.963548,46.515144,43.878494,NaN,NaN,32.974386,31.421021


In [53]:
# Check date ranges for saved food price feature files

saved_county_prices["month"] = pd.to_datetime(saved_county_prices["month"], errors="coerce")
saved_national_prices["month"] = pd.to_datetime(saved_national_prices["month"], errors="coerce")

print("County food price feature date range:")
print("Start:", saved_county_prices["month"].min())
print("End:", saved_county_prices["month"].max())

print("\nNational food price feature date range:")
print("Start:", saved_national_prices["month"].min())
print("End:", saved_national_prices["month"].max())

print("\nProject analysis period:")
print("Start:", analysis_start)
print("End:", analysis_end)

County food price feature date range:
Start: 2006-01-01 00:00:00
End: 2026-04-01 00:00:00

National food price feature date range:
Start: 2006-01-01 00:00:00
End: 2026-04-01 00:00:00

Project analysis period:
Start: 2019-07-01 00:00:00
End: 2026-02-01 00:00:00


In [54]:
# Check coverage during the project analysis period

county_saved_analysis_period = saved_county_prices[
    (saved_county_prices["month"] >= analysis_start) &
    (saved_county_prices["month"] <= analysis_end)
].copy()

national_saved_analysis_period = saved_national_prices[
    (saved_national_prices["month"] >= analysis_start) &
    (saved_national_prices["month"] <= analysis_end)
].copy()

print("County saved analysis-period shape:", county_saved_analysis_period.shape)
print("National saved analysis-period shape:", national_saved_analysis_period.shape)

print("\nCounty analysis-period months:", county_saved_analysis_period["month"].nunique())
print("National analysis-period months:", national_saved_analysis_period["month"].nunique())

County saved analysis-period shape: (540, 16)
National saved analysis-period shape: (80, 13)

County analysis-period months: 80
National analysis-period months: 80


## **Food Price Feature Files Saved Successfully**

The processed food price feature files were saved successfully.

Two files were created:

- `county_food_price_monthly_features.csv`
- `national_food_price_monthly_features.csv`

Both files cover the project analysis period from July 2019 to February 2026.

The county-level feature file contains 540 rows during the analysis period, while the national-level feature file contains 80 monthly rows during the same period.

This confirms that the food price features are ready for the next step: merging with the IPC + rainfall + NDVI master modeling dataset.